# Weighted Object-Centric Alignment Testing

This notebook demonstrates the new weighted alignment functionality with activity-specific weights.

## Overview
1. **Part 1**: Load data (OCPN jsonocel, single trace jsonocel, activity weights)
2. **Part 2**: Standard alignment (baseline)
3. **Part 3**: Weighted alignment with zero weights (should match Part 2)
4. **Part 4**: Weighted alignment with actual weights
5. **Part 5**: Custom formula alignment (weight * 100)

---
## Part 1: Data Loading

Load OCEL files and weights, discover OCPN from full log.

In [6]:
# Setup path
# import sys
# sys.path.insert(0, '/home/user/ocpa')

import json
import matplotlib.pyplot as plt
%matplotlib inline

# Import OCPA
from ocpa.objects.log.importer.ocel import factory as ocel_import_factory
from ocpa.algo.discovery.ocpn import algorithm as ocpn_discovery_factory
from ocpa.visualization.oc_petri_net import factory as ocpn_vis_factory

# Import alignment functions
from ocpa.algo.conformance.alignments import algorithm as alignment_factory
from ocpa.algo.conformance.alignments.alignment import (
    create_weighted_cost_function,
    create_custom_cost_function,
    CostFunction
)

# Import visualization
from ocpa.visualization.alignment_viz.visualization import alignment_viz
import pickle

print("✓ All imports successful")

✓ All imports successful


In [7]:
# Load full OCEL for OCPN discovery
print("Loading full OCEL for OCPN discovery...")
ocel_full = ocel_import_factory.apply("./data/BPIC17.jsonocel")
print(f"  Events: {len(ocel_full.log.log)}")
# print(f"  Objects: {len(ocel_full.objects)}")
# print(f"  Object types: {ocel_full.object_types}")

Loading full OCEL for OCPN discovery...
  Events: 1202267


In [9]:
# Discover OCPN
print("\nDiscovering OCPN...")
ocpn = ocpn_discovery_factory.apply(ocel_full, parameters={"debug": False})
print(f"  Places: {len(ocpn.places)}")
print(f"  Transitions: {len(ocpn.transitions)}")
print(f"  Arcs: {len(ocpn.arcs)}")

print("\n✓ OCPN discovered successfully")


Discovering OCPN...
  Places: 81
  Transitions: 121
  Arcs: 296

✓ OCPN discovered successfully


In [10]:
type(ocel_full), type(ocpn)

(ocpa.objects.log.ocel.OCEL,
 ocpa.objects.oc_petri_net.obj.ObjectCentricPetriNet)

In [2]:
import pickle

In [11]:
with open('./objects/ocel_object.pkl', 'wb+') as f:
    pickle.dump(ocel_full, f)

In [12]:
with open('./objects/ocpn_object.pkl', 'wb+') as f:
    pickle.dump(ocpn, f)

In [8]:
with open('./objects/ocel_object.pkl', 'rb') as f:
    ocel_full = pickle.load(f)
with open('./objects/ocpn_object.pkl', 'rb') as f:
    ocpn = pickle.load(f)

EOFError: Ran out of input

In [ ]:
# Load single trace OCEL for alignment
print("Loading single trace OCEL...")
ocel_trace = ocel_import_factory.apply("./data/BPIC17__case_Application_2062326086.jsonocel")
print(f"  Events: {len(ocel_trace.log.log)}")
# print(f"  Objects: {len(ocel_trace.objects)}")
# print(f"  Variants: {len(ocel_trace.variants)}")

# Show trace details
# variant_id = list(ocel_trace.variants.keys())[0]
# print(f"\nVariant ID: {variant_id}")
# print(f"Number of process executions: {len(ocel_trace.variants_dict[variant_id])}")

In [ ]:
# Load activity weights
print("Loading activity weights...")

with open("data/BPIC17_unified_equal.json", "r") as f:
    weights_equal = json.load(f)

with open("data/BPIC17_unified.json", "r") as f:
    weights_varied = json.load(f)

print("\nEqual weights (all activities):")
for act, weight in sorted(weights_equal.items())[:5]:
    print(f"  {act:<25} {weight:.4f}")
print(f"  ... ({len(weights_equal)} activities total)")

print("\nVaried weights (top 5):")
for act, weight in sorted(weights_varied.items(), key=lambda x: -x[1])[:5]:
    print(f"  {act:<25} {weight:.4f}")

print(f"\n✓ Weights loaded (sum = {sum(weights_varied.values()):.6f})")

---
## Part 2: Standard Alignment (Baseline)

Calculate alignment using the original `calculate_oc_alignments()` function.

In [ ]:
print("="*70)
print("PART 2: Standard Alignment (Baseline)")
print("="*70)

# Calculate standard alignment
print("\nCalculating standard alignment...")
alignments_standard = alignment_factory.calculate_oc_alignments(ocel_trace, ocpn)

# Get the alignment for our variant
# variant_id = list(alignments_standard.keys())[0]
# alignment_standard = alignments_standard[variant_id]

# print(f"\nAlignment computed:")
# print(f"  Variant ID: {variant_id}")
# print(f"  Number of moves: {len(alignment_standard.moves)}")
# print(f"  Total cost: {alignment_standard.get_cost():.2f}")

In [ ]:
type(alignments_standard), alignments_standard.keys()

In [ ]:
alignment_standard = alignments_standard['3042efc33354cf43f4026cbe3174abab']

In [ ]:
print(type(alignment_standard))

In [ ]:
alignment_standard.moves

In [ ]:
# Show move details
print("\nMove details:")
print(f"{'#':<4} {'Type':<15} {'Log Move':<25} {'Model Move':<25} {'Objects':<10} {'Cost':>8}")
print("-" * 100)

for i, move in enumerate(alignment_standard.moves, 1):
    move_type = "Sync" if (move.log_move and move.model_move) else \
                ("Log" if move.log_move else "Model")
    log_act = move.log_move or "--"
    model_act = move.model_move or "--"
    num_objs = len(move.objects) if move.objects else 0
    
    print(f"{i:<4} {move_type:<15} {log_act:<25} {model_act:<25} {num_objs:<10} {move.cost:>8.2f}")

In [ ]:
# Visualize alignment
print("\nVisualizing standard alignment...")
plt_standard = alignment_viz(alignment_standard)
plt_standard.gcf().set_size_inches(15, 8)
plt_standard.title(f"Part 2: Standard Alignment (Cost: {alignment_standard.get_cost():.2f})")
plt_standard.tight_layout()
plt_standard.show()

print("✓ Standard alignment visualization complete")

---
## Part 3: Weighted Alignment with Zero Weights

Test weighted alignment with all weights set to 0. This should produce identical results to Part 2.

In [ ]:
print("="*70)
print("PART 3: Weighted Alignment with Zero Weights")
print("="*70)

# Create zero weights dictionary
weights_zero = {act: 0.0 for act in weights_equal.keys()}

print("\nZero weights (all activities have weight 0.0):")xw
print(f"  Total activities: {len(weights_zero)}")
print(f"  All weights: 0.0")
print(f"  Formula: cost = len(objects) * (1 + 0) = len(objects)")
print(f"  Expected: Same as standard alignment")

In [ ]:
# Calculate weighted alignment with zero weights
print("\nCalculating weighted alignment (weight=0)...")
alignments_zero = alignment_factory.calculate_oc_alignments_with_weights(
    ocel_trace, ocpn, weights_zero
)

alignment_zero = alignments_zero[variant_id]

print(f"\nAlignment computed:")
print(f"  Number of moves: {len(alignment_zero.moves)}")
print(f"  Total cost: {alignment_zero.get_cost():.2f}")

In [ ]:
# Compare with standard alignment
print("\nComparison with standard alignment:")
print(f"  Standard cost:  {alignment_standard.get_cost():.2f}")
print(f"  Zero-weight cost: {alignment_zero.get_cost():.2f}")
print(f"  Difference: {abs(alignment_standard.get_cost() - alignment_zero.get_cost()):.6f}")

if abs(alignment_standard.get_cost() - alignment_zero.get_cost()) < 0.001:
    print("\n  ✓ PASS: Costs match (as expected)")
else:
    print("\n  ✗ FAIL: Costs don't match (unexpected!)")

# Check if moves match
print(f"\n  Number of moves - Standard: {len(alignment_standard.moves)}, Zero-weight: {len(alignment_zero.moves)}")
if len(alignment_standard.moves) == len(alignment_zero.moves):
    print("  ✓ PASS: Number of moves match")
else:
    print("  ✗ FAIL: Number of moves don't match")

In [ ]:
# Visualize zero-weight alignment
print("\nVisualizing zero-weight alignment...")
plt_zero = alignment_viz(alignment_zero)
plt_zero.gcf().set_size_inches(15, 8)
plt_zero.title(f"Part 3: Zero-Weight Alignment (Cost: {alignment_zero.get_cost():.2f})")
plt_zero.tight_layout()
plt_zero.show()

print("✓ Zero-weight alignment visualization complete")

---
## Part 4: Weighted Alignment with Varied Weights

Apply actual activity-specific weights to see how they affect alignment costs.

In [ ]:
print("="*70)
print("PART 4: Weighted Alignment with Varied Weights")
print("="*70)

print("\nApplying varied activity weights:")
print("  Formula: cost = len(objects) * (1 + weight)")
print("\nTop 5 weighted activities:")
for act, weight in sorted(weights_varied.items(), key=lambda x: -x[1])[:5]:
    print(f"  {act:<25} weight={weight:.4f}")

In [ ]:
# Calculate weighted alignment with varied weights
print("\nCalculating weighted alignment (varied weights)...")
alignments_varied = alignment_factory.calculate_oc_alignments_with_weights(
    ocel_trace, ocpn, weights_varied
)

alignment_varied = alignments_varied[variant_id]

print(f"\nAlignment computed:")
print(f"  Number of moves: {len(alignment_varied.moves)}")
print(f"  Total cost: {alignment_varied.get_cost():.2f}")

In [ ]:
# Compare with standard alignment
print("\nComparison with standard alignment:")
print(f"  Standard cost:    {alignment_standard.get_cost():.2f}")
print(f"  Weighted cost:    {alignment_varied.get_cost():.2f}")
print(f"  Difference:       {alignment_varied.get_cost() - alignment_standard.get_cost():.2f}")
print(f"  Percent increase: {((alignment_varied.get_cost() / alignment_standard.get_cost()) - 1) * 100:.2f}%")

print("\nInterpretation:")
if alignment_varied.get_cost() > alignment_standard.get_cost():
    print("  Weighted cost is HIGHER - deviations involve important activities")
elif alignment_varied.get_cost() < alignment_standard.get_cost():
    print("  Weighted cost is LOWER - deviations involve less important activities")
else:
    print("  Costs are EQUAL - no deviations or weights don't affect this trace")

In [ ]:
# Show move-by-move comparison
print("\nMove-by-move cost comparison:")
print(f"{'#':<4} {'Type':<10} {'Activity':<25} {'Objects':<8} {'Standard':>10} {'Weighted':>10} {'Diff':>8}")
print("-" * 100)

for i, (move_std, move_var) in enumerate(zip(alignment_standard.moves, alignment_varied.moves), 1):
    move_type = "Sync" if (move_std.log_move and move_std.model_move) else \
                ("Log" if move_std.log_move else "Model")
    activity = move_std.log_move or move_std.model_move or "--"
    num_objs = len(move_std.objects) if move_std.objects else 0
    diff = move_var.cost - move_std.cost
    
    print(f"{i:<4} {move_type:<10} {activity:<25} {num_objs:<8} {move_std.cost:>10.2f} {move_var.cost:>10.2f} {diff:>8.2f}")

In [ ]:
# Visualize weighted alignment
print("\nVisualizing weighted alignment...")
plt_varied = alignment_viz(alignment_varied)
plt_varied.gcf().set_size_inches(15, 8)
plt_varied.title(f"Part 4: Weighted Alignment (Cost: {alignment_varied.get_cost():.2f})")
plt_varied.tight_layout()
plt_varied.show()

print("✓ Weighted alignment visualization complete")

---
## Part 5: Custom Formula Alignment (Weight × 100)

Use a custom cost formula to amplify the weight effect.

In [ ]:
print("="*70)
print("PART 5: Custom Formula Alignment (Weight × 100)")
print("="*70)

print("\nCustom formula:")
print("  cost = len(objects) * (1 + weight * 100)")
print("\nThis amplifies the weight effect by 100x")
print("\nExample calculations:")
print("  Activity with weight=0.25 and 2 objects:")
print("    Standard: 2 * (1 + 0.25) = 2.5")
print("    Custom:   2 * (1 + 0.25*100) = 2 * 26 = 52.0")

In [ ]:
# Create custom cost function
custom_formula = lambda num_obj, weight: num_obj * (1 + weight * 100)

print("\nCreating custom cost function...")
custom_cost_fn_creator = create_custom_cost_function(custom_formula)
custom_cost_fn = custom_cost_fn_creator(weights_varied)

print("✓ Custom cost function created")

In [ ]:
# Calculate alignment with custom cost function
print("\nCalculating alignment with custom cost function...")
alignments_custom = alignment_factory.calculate_oc_alignments_with_cost_function(
    ocel_trace, ocpn, custom_cost_fn
)

alignment_custom = alignments_custom[variant_id]

print(f"\nAlignment computed:")
print(f"  Number of moves: {len(alignment_custom.moves)}")
print(f"  Total cost: {alignment_custom.get_cost():.2f}")

In [ ]:
# Compare all three approaches
print("\nComparison of all alignment approaches:")
print(f"  Standard alignment:        {alignment_standard.get_cost():>10.2f}")
print(f"  Weighted alignment:        {alignment_varied.get_cost():>10.2f}  (+{alignment_varied.get_cost() - alignment_standard.get_cost():.2f})")
print(f"  Custom (×100) alignment:   {alignment_custom.get_cost():>10.2f}  (+{alignment_custom.get_cost() - alignment_standard.get_cost():.2f})")

print("\nCost ratios:")
print(f"  Weighted / Standard:       {alignment_varied.get_cost() / alignment_standard.get_cost():.2f}x")
print(f"  Custom / Standard:         {alignment_custom.get_cost() / alignment_standard.get_cost():.2f}x")
print(f"  Custom / Weighted:         {alignment_custom.get_cost() / alignment_varied.get_cost():.2f}x")

In [ ]:
# Show detailed cost comparison
print("\nDetailed cost comparison (first 10 moves):")
print(f"{'#':<4} {'Activity':<25} {'Objects':<8} {'Standard':>10} {'Weighted':>10} {'Custom':>10}")
print("-" * 90)

for i, (move_std, move_var, move_cst) in enumerate(
    zip(alignment_standard.moves[:10], alignment_varied.moves[:10], alignment_custom.moves[:10]), 1
):
    activity = move_std.log_move or move_std.model_move or "--"
    num_objs = len(move_std.objects) if move_std.objects else 0
    
    print(f"{i:<4} {activity:<25} {num_objs:<8} {move_std.cost:>10.2f} {move_var.cost:>10.2f} {move_cst.cost:>10.2f}")

In [ ]:
# Visualize custom alignment
print("\nVisualizing custom formula alignment...")
plt_custom = alignment_viz(alignment_custom)
plt_custom.gcf().set_size_inches(15, 8)
plt_custom.title(f"Part 5: Custom Formula Alignment (Cost: {alignment_custom.get_cost():.2f})")
plt_custom.tight_layout()
plt_custom.show()

print("✓ Custom formula alignment visualization complete")

---
## Summary

Compare all five alignment results side by side.

In [ ]:
print("="*70)
print("SUMMARY: Weighted Alignment Comparison")
print("="*70)

print("\nFinal Cost Comparison:")
print(f"  1. Standard (baseline):    {alignment_standard.get_cost():>10.2f}")
print(f"  2. Zero weights:           {alignment_zero.get_cost():>10.2f}  (should equal #1)")
print(f"  3. Varied weights:         {alignment_varied.get_cost():>10.2f}  (formula: len(obj) * (1 + w))")
print(f"  4. Custom (w×100):         {alignment_custom.get_cost():>10.2f}  (formula: len(obj) * (1 + w*100))")

print("\nKey Findings:")
print(f"  • Zero weights match standard: {abs(alignment_zero.get_cost() - alignment_standard.get_cost()) < 0.001}")
print(f"  • Varied weights change cost: {abs(alignment_varied.get_cost() - alignment_standard.get_cost()) > 0.001}")
print(f"  • Custom formula amplifies effect: {alignment_custom.get_cost() / alignment_varied.get_cost():.1f}x")

print("\n" + "="*70)
print("Weighted alignment testing completed successfully!")
print("="*70)